# 02 — PMSM Inverter Fault Data Audit

**HealthCar · Phase 2: Labeled laboratory fault-data audit**

This notebook describes **laboratory PMSM/inverter faults**, not real-world BEV fault labels. Run `python scripts/download_data.py --only pmsm` first. No training until labels, classes and session boundaries are understood.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
assert (ROOT / "src").is_dir()
sys.path.insert(0, str(ROOT / "src"))
from healthcar.audit import read_dataset, audit, write_report
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print("Root:", ROOT)

## 1. Inspect actual published files and labels

In [ ]:
df, source, candidates = read_dataset("pmsm", None, ROOT / "data" / "raw")
print("Selected:", source, "\nShape:", df.shape)
display(pd.DataFrame(candidates[:8]))
display(df.head())
print("Columns:", list(df.columns))

## 2. Quantify label distribution and data problems

In [ ]:
report, _ = write_report("pmsm", None,
    raw_root=ROOT / "data" / "raw",
    reports_dir=ROOT / "data" / "reports")
print("Label candidates:", report.get("label_candidates"))
print("Potential sensor columns:", report.get("sensor_candidates"))
display(pd.DataFrame({
    "column": df.columns,
    "dtype": [str(t) for t in df.dtypes],
    "missing_percent": [report["missing_percent"].get(c) for c in df.columns]
}))
for name, counts in report.get("class_counts",{}).items():
    display(pd.Series(counts, name="observations").to_frame())

## 3. Explore classes and original sensor channels

A high score after a random row split may be misleading because neighboring measurements from the same experimental run can be extremely similar.

In [ ]:
label_cols = report.get("label_candidates", [])
if label_cols:
    target = label_cols[0]
    counts = df[target].value_counts(dropna=False)
    plt.figure(figsize=(9,3))
    counts.plot(kind="bar")
    plt.title(f"Class distribution — {target}")
    plt.ylabel("Observations")
    plt.tight_layout()
    plt.show()
else:
    print("Target not detected automatically — inspect files before modeling.")
sensors = [c for c in df.columns if c.lower() in {"ia","ib","vdc","idc","t1","t2","t3","vd"}]
if sensors:
    display(df[sensors].apply(pd.to_numeric, errors="coerce").describe().T)

## 4. Checklist before classification

- Confirm F0–F8 definitions and whether the target column is trustworthy.
- Find recording/session identifiers or a safe temporal grouping method.
- Detect any features derived directly from target or fault injection.
- Establish a majority-class baseline before Random Forest.
- Report per-class recall, macro-F1 and confusion matrix only on a leakage-aware test.
- Explicitly label all results **laboratory rig**, not on-road EV prediction.